# Day 1 — Data Curation
Loads the raw used-cars CSV, cleans it, builds `Car` pydantic objects, splits into train/val/test, and pushes to Hugging Face Hub.

**Toggle `LITE_MODE` to switch between the 50k lite run and the 500k full run.**

## 0. Config & Imports

In [17]:
# --- Toggle this flag to switch between lite (50k) and full (500k) runs ---
LITE_MODE = False

HF_USER    = "ShahidHKhan"
NROWS      = 50_000   if LITE_MODE else 500_000
DATASET_NAME = f"{HF_USER}/cars_lite_raw" if LITE_MODE else f"{HF_USER}/cars_full_raw"

print(f"Mode: {'LITE' if LITE_MODE else 'FULL'}")
print(f"Rows to load: {NROWS:,}")
print(f"Target dataset: {DATASET_NAME}")

Mode: FULL
Rows to load: 500,000
Target dataset: ShahidHKhan/cars_full_raw


In [18]:
import os
import re
import sys
import random

import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login

sys.path.append("..")
from auto_pricer.car import Car

load_dotenv("../.env")
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)
print("Imports done, HF logged in.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Imports done, HF logged in.


## 1. Load Raw CSV

In [19]:
df = pd.read_csv("../data/used_cars_data.csv", nrows=NROWS, low_memory=False)
print(f"Loaded: {df.shape}")

Loaded: (500000, 66)


## 2. Drop Dead Columns
Columns confirmed as 100% null, truck-specific, or pure metadata with no predictive value.

In [20]:
dead_columns = [
    # 100% null
    "is_certified", "combine_fuel_economy", "vehicle_damage_category",
    # truck-specific (>90% null)
    "bed", "cabin", "bed_length", "bed_height",
    # listing/dealer metadata — no predictive value
    "is_oemcpo", "isCab", "main_picture_url",
    "listing_id", "sp_id", "sp_name", "dealer_zip",
    "latitude", "longitude", "vin",
]

df = df.drop(columns=dead_columns)
print(f"After dropping dead columns: {df.shape}")

After dropping dead columns: (500000, 49)


## 3. Drop New Cars
This is a used-car pricer — new-car listings follow MSRP/dealer-markup logic, not used-car depreciation signals.

In [21]:
print(df["is_new"].value_counts(dropna=False))
df = df[df["is_new"] == False]
print(f"After dropping new cars: {df.shape}")

is_new
False    262001
True     237999
Name: count, dtype: int64
After dropping new cars: (262001, 49)


## 4. Impute Missing Values
### 4a. Damage/history cluster → False
These 5 columns share the same ~40% missingness pattern (one underlying vehicle-history-report source).
Imputing `False` is safe: sellers with clean histories are more likely to have the report available.

In [22]:
damage_cols = ["salvage", "theft_title", "frame_damaged", "fleet", "has_accidents"]
df[damage_cols] = df[damage_cols].fillna(False)
print("Remaining nulls in damage cluster:")
print(df[damage_cols].isna().sum())

Remaining nulls in damage cluster:
salvage          0
theft_title      0
frame_damaged    0
fleet            0
has_accidents    0
dtype: int64


### 4b. `owner_count` → median

In [23]:
median_owners = df["owner_count"].median()
df["owner_count"] = df["owner_count"].fillna(median_owners)
print(f"Median owner count: {median_owners}")
print(f"Remaining nulls: {df['owner_count'].isna().sum()}")

Median owner count: 1.0
Remaining nulls: 0


### 4c. Numeric columns → median
Note: `torque` and `power` are stored as composite strings (e.g. `'200 lb-ft @ 1,750 RPM'`) — extract the leading number before imputing.

In [24]:
# Extract numeric part from composite string columns
df["torque"] = df["torque"].str.extract(r"([\d.]+)").astype(float)
df["power"]  = df["power"].str.extract(r"([\d.]+)").astype(float)

numeric_cols = [
    "horsepower", "torque", "power",
    "highway_fuel_economy", "city_fuel_economy",
    "engine_displacement",
]
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print("Remaining nulls in numeric columns:")
print(df[numeric_cols].isna().sum())

Remaining nulls in numeric columns:
horsepower              0
torque                  0
power                   0
highway_fuel_economy    0
city_fuel_economy       0
engine_displacement     0
dtype: int64


## 5. Drop Rows Missing Critical Fields
No sensible imputation exists for price, mileage, year, make, or model — these rows are unusable.

In [25]:
critical_cols = ["price", "mileage", "year", "make_name", "model_name"]
print("Nulls in critical columns:")
print(df[critical_cols].isna().sum())

df = df.dropna(subset=critical_cols)
print(f"\nShape after drop: {df.shape}")

Nulls in critical columns:
price            0
mileage       3068
year             0
make_name        0
model_name       0
dtype: int64

Shape after drop: (258933, 49)


## 6. Filter Outliers

In [26]:
before = len(df)

df = df[df["price"].between(500, 150_000)]  # excludes salvage-only and exotic outliers
df = df[df["year"] >= 1990]                 # pre-1990 cars are collectibles, not daily drivers
df = df[df["mileage"] <= 300_000]           # beyond this, price signal collapses

print(f"Rows before: {before:,}")
print(f"Rows after:  {len(df):,}")
print(f"Dropped:     {before - len(df):,}")

Rows before: 258,933
Rows after:  257,719
Dropped:     1,214


## 7. Build `full` Text Blob
Combines structured fields + truncated raw description into one text field.
This is what Gemini will rewrite into clean summaries in Day 2.

In [27]:
def build_full(row):
    desc = str(row["description"]) if pd.notna(row["description"]) else ""
    desc = re.sub(r'\[!@@.*?@@!\]', '', desc).strip()  # strip cars.com scraping artifacts
    desc = desc[:400]  # cap to stop options-lists from dominating

    return (
        f"{row['year']} {row['make_name']} {row['model_name']} {row['trim_name']}\n"
        f"Body: {row['body_type']} | Mileage: {row['mileage']:,.0f} mi\n"
        f"Horsepower: {row['horsepower']} | Fuel: {row['fuel_type']}\n"
        f"Transmission: {row['transmission_display']}\n"
        f"Accidents: {row['has_accidents']} | Frame damaged: {row['frame_damaged']}\n"
        f"Description: {desc}"
    )

df["full"] = df.apply(build_full, axis=1)

# Sanity check — print one example
print(df["full"].iloc[0])

2020 Land Rover Range Rover Evoque P300 R-Dynamic SE AWD
Body: SUV / Crossover | Mileage: 254 mi
Horsepower: 296.0 | Fuel: Gasoline
Transmission: 9-Speed Automatic Overdrive
Accidents: False | Frame damaged: False
Description: 4-Wheel Disc Brakes,A/C,ABS,Adjustable Steering Wheel,All Wheel Drive,Aluminum Wheels,Auto-Dimming Rearview Mirror,Automatic Headlights,Automatic Parking,Auxiliary Audio Input,Back-Up Camera,Bluetooth Connection,Brake Actuated Limited Slip Differential,Brake Assist,Bucket Seats,Cargo Shade,Child Safety Locks,Climate Control,Cross-Traffic Alert,Cruise Control,Daytime Running Lights,Driver Adjustabl


## 8. Build `Car` Objects

In [28]:
cars = []
for _, row in df.iterrows():
    car = Car(
        title=f"{row['year']} {row['make_name']} {row['model_name']}",
        make=row["make_name"],
        model=row["model_name"],
        trim=row["trim_name"] if pd.notna(row["trim_name"]) else None,
        year=int(row["year"]),
        mileage=float(row["mileage"]),
        price=float(row["price"]),
        horsepower=float(row["horsepower"]) if pd.notna(row["horsepower"]) else None,
        body_type=row["body_type"] if pd.notna(row["body_type"]) else None,
        fuel_type=row["fuel_type"] if pd.notna(row["fuel_type"]) else None,
        transmission=row["transmission_display"] if pd.notna(row["transmission_display"]) else None,
        has_accidents=bool(row["has_accidents"]),
        frame_damaged=bool(row["frame_damaged"]),
        full=row["full"],
        id=len(cars),
    )
    cars.append(car)

print(f"Total Car objects: {len(cars):,}")
print(repr(cars[0]))

Total Car objects: 257,719
<2020 Land Rover Range Rover Evoque = $84399.0>


## 9. Train / Val / Test Split (90 / 5 / 5)

In [29]:
random.seed(42)
random.shuffle(cars)

n         = len(cars)
train_end = int(n * 0.90)
val_end   = int(n * 0.95)

train = cars[:train_end]
val   = cars[train_end:val_end]
test  = cars[val_end:]

print(f"Train: {len(train):,}")
print(f"Val:   {len(val):,}")
print(f"Test:  {len(test):,}")
print(f"Total: {len(train)+len(val)+len(test):,}")

Train: 231,947
Val:   12,886
Test:  12,886
Total: 257,719


## 10. Push to Hugging Face Hub

In [30]:
Car.push_to_hub("ShahidHKhan/cars_full_raw", train, val, test)
print(f"Pushed to: ShahidHKhan/cars_full_raw")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  2.26ba/s]
Processing Files (1 / 1): 100%|██████████| 69.1MB / 69.1MB, 6.43MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.28s/ shards]
Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 20.05ba/s]
Processing Files (1 / 1): 100%|██████████| 3.91MB / 3.91MB,  382kB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.77 shards/s]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 19.53

Pushed to: ShahidHKhan/cars_full_raw


## 11. Verify Round-Trip from Hub

In [31]:
train_check, val_check, test_check = Car.from_hub("ShahidHKhan/cars_full_raw")
print(f"Train: {len(train_check):,}, Val: {len(val_check):,}, Test: {len(test_check):,}")
print(repr(train_check[0]))

Generating test split: 100%|██████████| 12886/12886 [00:00<00:00, 464811.37 examples/s]


Train: 231,947, Val: 12,886, Test: 12,886
<2013 Honda CR-V = $17995.0>


In [32]:
print(f"LITE_MODE: {LITE_MODE}")
print(f"NROWS: {NROWS}")
print(f"DATASET_NAME: {DATASET_NAME}")

LITE_MODE: False
NROWS: 500000
DATASET_NAME: ShahidHKhan/cars_full_raw
